# Micro-C compartments — inter-NPR spacing ECDF (chr1)

In [ ]:
from pathlib import Path

# Resolve the repo root from this notebook's location (repo_root/notebooks/this_file.ipynb)
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

import pyBigWig as pbw
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl_style = REPO_ROOT / "notebooks" / "genome_research.mplstyle"
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.style.use(mpl_style)

plt.rcParams.update({
    # Font — picks the first one it finds installed
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],

    # Force every text element to the same size
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "figure.titlesize": 10,

    # Keep text as real, editable text in the SVG — not outlines
    "svg.fonttype": "none",
})

import sys
sys.path.insert(1, str(REPO_ROOT / "src"))
import utils as mf
from sklearn.decomposition import TruncatedSVD
from scipy import sparse

DATA_DIR = REPO_ROOT / "data"
DEMO_DIR = REPO_ROOT / "demo"

PATHS = {
    # same key/location as in figures_new_clean.ipynb
    "chrom_sizes": DATA_DIR / "sequences" / "mm10.chrom.sizes",
    "crossmut_bw": DEMO_DIR / "test" / "crossmut.bw",
    "fig_path":    REPO_ROOT / "notebooks" / "FIGTEST",
    # new for this notebook - place the Micro-C intrachromosomic contact files here
    "microc_intrachrom_dir": DATA_DIR / "microc_intrachromosomic",
    "nprs": DEMO_DIR / "test" / "nprs.csv"
}
PATHS["fig_path"].mkdir(parents=True, exist_ok=True)


def kth_diag_indices(a, k):
    rows, cols = np.diag_indices_from(a)
    if k < 0:
        return rows[-k:], cols[:k]
    elif k > 0:
        return rows[:-k], cols[k:]
    else:
        return rows, cols


def get_num(a):
    s = ""
    for c in a:
        if c.isnumeric():
            s += c
    return int(s)


def get_num_vec(a):
    return a.apply(get_num)


chrom_sizes = pd.read_csv(PATHS["chrom_sizes"], header=None, sep="\t")
chrom_sizes = chrom_sizes[np.isin(chrom_sizes[0], [f"chr{i}" for i in range(1, 20)])]
chrom_sizes = chrom_sizes.sort_values(by=0, key=get_num_vec)


def scn(a, iter_max=3):
    for i in range(iter_max):
        a /= a.sum(axis=0)
        a /= a.sum(axis=1)
    return (a + a.T) / 2


## A/B compartments — chr1 contact matrix and PCA (PC1)

In [ ]:
bin_size = 50000
nchrom = 1
df = pd.read_csv(PATHS["microc_intrachrom_dir"] / f"chr{nchrom}_coord.tsv", sep=" ", header=None).iloc[:-1, :]
df_binned = np.floor(df/bin_size).astype(int)
df_binned = df_binned.loc[np.diff(df_binned, axis=1)!=0]
chrom_s = chrom_sizes.iloc[nchrom-1, 1]

contact = sparse.csr_matrix(
    (np.ones(df_binned.shape[0]*2),
     (np.hstack([df_binned[0].values, df_binned[1].values]),
      np.hstack([df_binned[1].values, df_binned[0].values]))),
    shape=(np.floor(chrom_s/bin_size).astype(int),)*2,
    dtype=np.int32
)


In [ ]:
binsum = np.array(contact.sum(axis=1)).ravel()
_ = plt.hist(binsum, bins=2000)
plt.yscale("log")
plt.xscale("log")
threshold = 1000
plt.axvline(threshold, color="k", linestyle="--")


In [ ]:
# FILTER
filter_idx = binsum >= threshold
contact_filtered = contact[filter_idx, :][:, filter_idx].astype(np.float32)

# SCN NORMALIZATION
scn_contact = scn(contact_filtered, 10)

# COMPARTMENTS (DIAGONAL MEAN)
arr = scn_contact.toarray()
for k in range(-len(arr) + 1, len(arr)):
    if k == 0:
        continue
    row, col = kth_diag_indices(arr, k)
    diag = np.diag(arr, k)
    arr[row, col] /= np.mean(diag + np.finfo(float).eps)

# CORRELATION
corrarr = np.corrcoef(np.nan_to_num(arr))
corrarr_diagnorm = sparse.csr_matrix(corrarr)

# PCA (first eigenvector = A/B compartment signal)
svd = TruncatedSVD(n_components=1)
svd.fit(np.nan_to_num(corrarr_diagnorm))

eigenvector = svd.components_[0]
z_eigen = np.zeros_like(binsum, dtype=float)
z_eigen[filter_idx] = eigenvector


In [ ]:
chromstate = z_eigen
chromstate = np.convolve(chromstate, np.ones(5)/5, mode="same")
plt.figure(figsize=(8, 4))
plt.fill_between(np.arange(len(chromstate)), chromstate, label='PC1')
plt.plot(np.arange(len(chromstate)), chromstate, label='PC1')
plt.xlabel("", fontsize=15, fontweight="bold")

x = np.arange(0, len(chromstate)+500, 500)
def bio_format(x, pos):
    """Format genomic positions in human-readable units."""
    if x >= 1e6:
        return f"{x/1e6:.1f} Mb"
    elif x >= 1e3:
        return f"{x/1e3:.1f} kb"
    else:
        return f"{int(x)} bp"

plt.xticks(
    ticks=x,
    labels=[bio_format(val*bin_size, None) for val in x],
    fontsize=12
)
plt.ylabel("Eigen values", fontsize=15, fontweight="bold")
plt.axhline(0, color="k")
plt.gca().spines[["top", "right", "bottom"]].set_visible(False)
plt.show()


## Signals on compartments — NPR calling (chr1)

In [ ]:
from scipy.signal import find_peaks

mut = pbw.open(str(PATHS["crossmut_bw"]))
mut1 = mut.values(f"chr{nchrom}", 0, -1, numpy=True)
nprs = pd.read_csv(PATHS["nprs"])
nprs = nprs[nprs.Chromosome==f"chr{nchrom}"].Start.values + 15
zmut = np.zeros_like(mut1)
zmut[nprs] = 1


def chromstate_plot(chromstate):
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    z = np.zeros_like(chromstate)
    z_p = np.argwhere(chromstate > 0)
    z[z_p] = 1
    plt.plot(z)
    plt.fill_between(np.arange(len(z)), z, alpha=0.3)

    z = np.zeros_like(chromstate)
    z_m = np.argwhere(chromstate < 0)
    z[z_m] = 1
    plt.plot(z)
    plt.fill_between(np.arange(len(z)), z, color="r", alpha=0.3)
    ax.spines[['top', 'right', "bottom"]].set_visible(False)
    ax.yaxis.set_visible(False)

    x = np.arange(0, len(chromstate)+500, 500)
    ax.set_xticks(
        ticks=x,
        labels=[bio_format(val*bin_size, None) for val in x],
    )
    return (fig, ax), z_p, z_m


(fig, ax), zp, zm = chromstate_plot(chromstate=chromstate)

ax2 = plt.twinx()

mut_binned = mf.reshape_bin(zmut, bin_size, np.sum)

plt.plot(mut_binned, color="k")

ax2.spines[['top', 'right', "bottom"]].set_visible(False)
ax2.yaxis.tick_left()
ax2.yaxis.set_label_position("left")
ax2.set_yticklabels([f"{x:.2f}" for x in ax2.get_yticks()])
ax2.set_ylabel("#NPR")


## Inter-NPR spacing ECDF, by compartment

In [ ]:
diffs_p = []
sl = [(x*bin_size)[[0, -1]] for x in mf.consecutive(zp.ravel(), 1)]
for s in sl:
    diffs_p.append(np.diff(nprs[(nprs >= s[0]) & (nprs < s[1])]))

diffs_m = []
sl = [(x*bin_size)[[0, -1]] for x in mf.consecutive(zm.ravel(), 1)]
for s in sl:
    diffs_m.append(np.diff(nprs[(nprs >= s[0]) & (nprs < s[1])]))


def ecdf(data):
    x = np.sort(data)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

plt.figure(figsize=(1.5, 1.5))
# condition +
x_p, y_p = ecdf(np.concatenate(diffs_p))
plt.plot(x_p, y_p, c="b", label="Comp. A")

# condition -
x_m, y_m = ecdf(np.concatenate(diffs_m))
plt.plot(x_m, y_m, c="r", label="Comp. B")

nb_nprs = pd.read_csv(PATHS["nprs"]).shape[0]
genome_size = chrom_sizes[1].sum()
avg_distance = genome_size/nb_nprs
x_rand, y_rand = ecdf((np.random.exponential(avg_distance, size=len(x_p))))
plt.plot(x_rand, y_rand, color="purple", label="Null mod.")

x_m, y_m = ecdf(np.diff(nprs))
plt.plot(x_m, y_m, c="k", linestyle=":", label="GW")

plt.legend(bbox_to_anchor=(1.05, 0.7))
# plt.xlabel("Distance")
# plt.ylabel("ECDF")
plt.xscale("log")
plt.xlim(20)
# plt.xticks([100, 1e3, 1e4])
plt.savefig(PATHS["fig_path"] / "cumulative_nprs_rdistance.svg")
plt.show()

## ΔECDF (observed − null exponential model)

In [ ]:
def ecdf(data):
    x = np.sort(data)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

# observed data
d_obs = np.concatenate(diffs_p)

# observed ECDF
x_obs, F_obs = ecdf(d_obs)

# fitted null exponential model
scale = np.mean(d_obs)
F_exp = 1 - np.exp(-x_obs / scale)

# ΔECDF
delta = F_obs - F_exp

plt.figure(figsize=(2.5, 2.5))
plt.plot(x_obs, delta, lw=2, c="b")
plt.axhline(0, ls="--", lw=1, c="k")
plt.axvline(168, ls="--", lw=1, c="k")

plt.xscale("log")

plt.xlabel("Inter-peak distance")
plt.ylabel("ΔECDF (obs − exp)")
plt.tight_layout()
